# Spine Annotator 브라우저용 모델 만들기
학습이 끝난 가중치 4종을 **한 번만 변환**합니다. 다시 학습하지 않습니다.
사용자는 변환된 폴더만 브라우저에서 선택하며, X-ray와 추론은 사용자 컴퓨터 안에 머뭅니다.
기존 학습 결과는 읽기만 합니다. 결과는 별도 새 폴더에 저장합니다.
개인 Drive 경로는 별도 설정 JSON에서 읽습니다. 모델 파일을 ChatGPT에 첨부할 필요 없이 이 Colab에서 읽어 변환합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 현재 웹페이지와 같은 소스 버전 사용
import io, zipfile, urllib.request, sys, subprocess
from pathlib import Path
SOURCE = Path('/content/spine_browser_converter')
SOURCE.mkdir(exist_ok=True)
url = 'https://github.com/kimtaejoon01/spine-annotator/archive/refs/heads/codex/integrated-ai-app.zip'
with urllib.request.urlopen(url) as response:
    archive = zipfile.ZipFile(io.BytesIO(response.read()))
    for item in archive.infolist():
        relative = Path(*Path(item.filename).parts[1:])
        if not relative.parts or item.is_dir(): continue
        target = (SOURCE / relative).resolve()
        if not target.is_relative_to(SOURCE.resolve()): raise ValueError('잘못된 압축 경로')
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(item))
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(SOURCE/'ai_server/browser-export-requirements.txt')])
sys.path.insert(0, str(SOURCE))
# 열린 런타임에서 재실행해도 이전 변환기가 남지 않도록 갱신합니다.
import importlib
importlib.invalidate_caches()
for module_name in ('ai_server.s1_pair', 'ai_server.onnx_export', 'ai_server.runtime', 'ai_server.export_browser'):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])


In [ ]:
# 경로 설정은 공개 저장소에 넣지 않고 개인 JSON에서 읽습니다.
# 처음 한 번 아래 선택창에 제공받은 spine_browser_model_paths.json을 넣으세요.
import json
from pathlib import Path
from google.colab import files
CONFIG_PATH = Path('/content/drive/MyDrive/spine_browser_model_paths.json')
if CONFIG_PATH.is_file():
    raw = CONFIG_PATH.read_bytes()
else:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('모델 경로 설정 JSON 파일 한 개만 선택하세요.')
    # Colab이 중복 파일명에 (1), (2) 등을 붙여도 내용을 읽습니다.
    raw = next(iter(uploaded.values()))
model_paths = json.loads(raw.decode('utf-8-sig'))
required = ('NUMBER_RUN', 'A_RUN', 'S1_BUNDLE', 'FH_RUN')
if not isinstance(model_paths, dict) or any(not isinstance(model_paths.get(k), str) or not model_paths[k].strip() for k in required):
    raise ValueError('네 모델 경로가 들어 있는 설정 파일이 필요합니다.')
if not CONFIG_PATH.is_file():
    CONFIG_PATH.write_bytes(raw)
NUMBER_RUN = model_paths['NUMBER_RUN']
A_RUN = model_paths['A_RUN']
S1_BUNDLE = model_paths['S1_BUNDLE']
FH_RUN = model_paths['FH_RUN']

from ai_server.runtime import normalize_path, read_json
from ai_server.s1_pair import resolve_s1_folder

def select_run(path, marker, kind):
    if not path.strip(): raise ValueError(kind+' 모델 폴더를 입력하세요.')
    root=normalize_path(path)
    if (root/marker).is_file(): candidates=[root]
    else: candidates=sorted({p.parent for p in root.rglob(marker)})
    if kind=='A': candidates=[p for p in candidates if read_json(p/marker).get('abc_variant')=='A' and (p/'checkpoints').is_dir()]
    if len(candidates)!=1:
        raise ValueError(kind+' 모델 폴더 한 개를 직접 지정하세요:\n'+'\n'.join(map(str,candidates)))
    return candidates[0]

paths=dict(number=select_run(NUMBER_RUN,'shared_mobilenet_roi_config.json','번호'),
           a=select_run(A_RUN,'config.json','A'),
           s1=resolve_s1_folder(normalize_path(S1_BUNDLE)),
           fh=select_run(FH_RUN,'coarse_best.pt','골두'))
for key,path in paths.items(): print(key, path)


In [ ]:
# 변환 + 원본 모델과 출력값 비교. CPU에서도 실행할 수 있습니다.
from datetime import datetime
from ai_server.export_browser import export_bundle
import uuid
OUTPUT = Path('/content/drive/MyDrive/spine_browser_models') / (datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8])
# 이전 A 변환이 수치 검사에서 멈췄다면 생성된 a.onnx 경로를 지정할 수 있습니다.
# 다시 내보내지 않고 현재 PyTorch 모델과 재비교한 뒤 새 폴더에 저장합니다.
REUSE_A_ONNX = None  # 예: Path('/content/drive/MyDrive/spine_browser_models/이전결과/a.onnx')
export_bundle(**paths, output=OUTPUT, reuse_a_onnx=REUSE_A_ONNX)
print('완료:', OUTPUT)


In [ ]:
# 내려받은 ZIP을 풀고, Annotator의 AI 분석 → 모델 폴더 선택에서 폴더 전체를 고릅니다.
import shutil
from google.colab import files
ZIP = shutil.make_archive('/content/spine_browser_models', 'zip', OUTPUT)
files.download(ZIP)


변환 중 오류가 나면 마지막 오류 메시지를 전달해 주세요. 검증을 통과한 경우에만 `manifest.json`이 생성됩니다.
브라우저에서도 모델을 처음 불러올 때 출력값을 확인합니다. 이 검사는 변환 일치성 검사이며 정확도 평가가 아닙니다.

A 확률지도는 최대·평균·99백분위 오차와 0.5 임계값 변경 비율을 함께 제한하는 `a-probability-v1` 기준을 씁니다. 번호·S1·골두의 기존 원소별 기준은 유지합니다. 웹앱도 최신 버전으로 배포해야 같은 A 검증 기준을 사용합니다.
